# FinanceBench Fast Hybrid Retrieval Evaluation

Notebook nay danh gia retrieval hien tai bang:
- Dense embedding nhanh: TF-IDF + TruncatedSVD 384 dimensions (`dense_lsa384`).
- Sparse search: TF-IDF lexical search (`sparse_tfidf`).
- Hybrid search: ket hop dense LSA va sparse TF-IDF.

Ly do co notebook fast: BGE neural embedding tren CPU cho 11,948 chunks rat cham. Notebook `financebench_current_retrieval_hybrid_eval.ipynb` van co BGE, con notebook nay cho ra ket qua nhanh de audit retrieval pipeline.

In [ ]:
from pathlib import Path
import json, math
from collections import defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import normalize
from sklearn.decomposition import TruncatedSVD

ROOT = Path('..').resolve()
CHUNKS_PATH = ROOT / 'outputs/financebench_eval/chunks.jsonl'
QUERIES_PATH = ROOT / 'outputs/financebench_eval/qrels/queries.jsonl'
QRELS_PATH = ROOT / 'outputs/financebench_eval/qrels/qrels.jsonl'
OUT_DIR = ROOT / 'outputs/financebench_eval/fast_hybrid_retrieval_eval'
OUT_DIR.mkdir(parents=True, exist_ok=True)

K_VALUES = [1, 3, 5, 10]
TOP_K = max(K_VALUES)
SVD_DIM = 384
HYBRID_ALPHA = 0.60

pd.set_option('display.max_colwidth', 160)

In [ ]:
def load_jsonl(path: Path) -> list[dict]:
    if not path.exists() or path.stat().st_size == 0:
        return []
    with path.open('r', encoding='utf-8') as f:
        return [json.loads(line) for line in f if line.strip()]

def write_jsonl(path: Path, rows: list[dict]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open('w', encoding='utf-8') as f:
        for row in rows:
            f.write(json.dumps(row, ensure_ascii=False) + '\n')

def chunk_text(chunk: dict) -> str:
    return str(chunk.get('text') or chunk.get('summary') or '')

def chunk_id(chunk: dict) -> str:
    return str(chunk.get('chunk_id') or chunk.get('id'))

def page_range(chunk: dict):
    start = chunk.get('page_start', chunk.get('page'))
    end = chunk.get('page_end', chunk.get('page'))
    try:
        start = int(start) if start is not None else None
        end = int(end) if end is not None else start
    except (TypeError, ValueError):
        return None, None
    return start, end

def minmax_row(scores: np.ndarray) -> np.ndarray:
    lo = scores.min(axis=1, keepdims=True)
    hi = scores.max(axis=1, keepdims=True)
    denom = np.where((hi - lo) == 0, 1.0, hi - lo)
    return (scores - lo) / denom

def topk_from_scores(score_matrix: np.ndarray, ids: list[str], queries: list[dict], chunks_by_id: dict[str, dict], top_k: int, method: str) -> list[dict]:
    rows = []
    for qi, query in enumerate(queries):
        scores = score_matrix[qi]
        candidate_idx = np.argpartition(-scores, top_k - 1)[:top_k]
        candidate_idx = candidate_idx[np.argsort(-scores[candidate_idx])]
        for rank, idx in enumerate(candidate_idx, start=1):
            cid = ids[int(idx)]
            chunk = chunks_by_id[cid]
            page_start, page_end = page_range(chunk)
            rows.append({
                'method': method,
                'query_id': query['query_id'],
                'rank': rank,
                'chunk_id': cid,
                'score': float(scores[int(idx)]),
                'doc_name': chunk.get('doc_name'),
                'page_start': page_start,
                'page_end': page_end,
                'chunk_type': chunk.get('chunk_type') or chunk.get('modality'),
                'text_preview': chunk_text(chunk)[:300],
            })
    return rows

## 1. Load retrieval benchmark

In [ ]:
chunks = load_jsonl(CHUNKS_PATH)
queries = load_jsonl(QUERIES_PATH)
qrels = load_jsonl(QRELS_PATH)

ids = [chunk_id(c) for c in chunks]
texts = [chunk_text(c) for c in chunks]
query_texts = [q['question'] for q in queries]
chunks_by_id = {chunk_id(c): c for c in chunks}

qrels_by_query = defaultdict(set)
for row in qrels:
    qrels_by_query[row['query_id']].add(row['chunk_id'])

print('chunks:', len(chunks))
print('queries:', len(queries))
print('qrels:', len(qrels))
print('queries with qrels:', len(qrels_by_query))

## 2. Build sparse + dense embedding scores

In [ ]:
vectorizer = TfidfVectorizer(
    lowercase=True,
    strip_accents='unicode',
    ngram_range=(1, 2),
    min_df=1,
    max_df=0.95,
    sublinear_tf=True,
    max_features=200000,
)
chunk_tfidf = normalize(vectorizer.fit_transform(texts), norm='l2', axis=1)
query_tfidf = normalize(vectorizer.transform(query_texts), norm='l2', axis=1)

sparse_scores = (query_tfidf @ chunk_tfidf.T).toarray().astype('float32')

svd = TruncatedSVD(n_components=SVD_DIM, random_state=42)
chunk_dense = svd.fit_transform(chunk_tfidf)
query_dense = svd.transform(query_tfidf)
chunk_dense = chunk_dense / (np.linalg.norm(chunk_dense, axis=1, keepdims=True) + 1e-12)
query_dense = query_dense / (np.linalg.norm(query_dense, axis=1, keepdims=True) + 1e-12)
dense_scores = (query_dense @ chunk_dense.T).astype('float32')

hybrid_scores = HYBRID_ALPHA * minmax_row(dense_scores) + (1.0 - HYBRID_ALPHA) * minmax_row(sparse_scores)

print('sparse_scores:', sparse_scores.shape)
print('dense_scores:', dense_scores.shape)
print('hybrid_scores:', hybrid_scores.shape)

## 3. Retrieve top-k

In [ ]:
dense_results = topk_from_scores(dense_scores, ids, queries, chunks_by_id, TOP_K, 'dense_lsa384')
sparse_results = topk_from_scores(sparse_scores, ids, queries, chunks_by_id, TOP_K, 'sparse_tfidf')
hybrid_results = topk_from_scores(hybrid_scores, ids, queries, chunks_by_id, TOP_K, f'hybrid_lsa_tfidf_alpha_{HYBRID_ALPHA:.2f}')

write_jsonl(OUT_DIR / 'dense_lsa384_retrieval_results.jsonl', dense_results)
write_jsonl(OUT_DIR / 'sparse_tfidf_retrieval_results.jsonl', sparse_results)
write_jsonl(OUT_DIR / f'hybrid_lsa_tfidf_alpha_{HYBRID_ALPHA:.2f}_retrieval_results.jsonl', hybrid_results)
write_jsonl(OUT_DIR / 'retrieval_results.jsonl', dense_results + sparse_results + hybrid_results)

len(dense_results), len(sparse_results), len(hybrid_results)

## 4. Compute Precision / Recall / Hit / MRR

In [ ]:
def reciprocal_rank(retrieved_ids: list[str], relevant_ids: set[str], k: int) -> float:
    for rank, cid in enumerate(retrieved_ids[:k], start=1):
        if cid in relevant_ids:
            return 1.0 / rank
    return 0.0

def evaluate_results(results: list[dict], method: str) -> tuple[dict, pd.DataFrame]:
    results_by_query = defaultdict(list)
    for row in results:
        results_by_query[row['query_id']].append(row)
    per_question = []
    for query in queries:
        qid = query['query_id']
        relevant_ids = qrels_by_query[qid]
        retrieved = sorted(results_by_query[qid], key=lambda row: row['rank'])
        retrieved_ids = [row['chunk_id'] for row in retrieved]
        row = {
            'method': method,
            'query_id': qid,
            'question': query['question'],
            'answer': query.get('answer'),
            'num_relevant_chunks': len(relevant_ids),
        }
        for k in K_VALUES:
            top_ids = retrieved_ids[:k]
            hits = [cid for cid in top_ids if cid in relevant_ids]
            row[f'precision@{k}'] = len(hits) / k
            row[f'recall@{k}'] = len(set(hits)) / len(relevant_ids)
            row[f'hit@{k}'] = 1.0 if hits else 0.0
            row[f'mrr@{k}'] = reciprocal_rank(retrieved_ids, relevant_ids, k)
        per_question.append(row)
    per_df = pd.DataFrame(per_question)
    summary = {'method': method, 'queries': len(queries)}
    for k in K_VALUES:
        for metric in ['precision', 'recall', 'hit', 'mrr']:
            summary[f'{metric}@{k}'] = float(per_df[f'{metric}@{k}'].mean())
    return summary, per_df

summaries = []
per_frames = []
for method, results in [
    ('dense_lsa384', dense_results),
    ('sparse_tfidf', sparse_results),
    (f'hybrid_lsa_tfidf_alpha_{HYBRID_ALPHA:.2f}', hybrid_results),
]:
    summary, per_df = evaluate_results(results, method)
    summaries.append(summary)
    per_frames.append(per_df)

summary_df = pd.DataFrame(summaries)
per_question_df = pd.concat(per_frames, ignore_index=True)
summary_df.to_json(OUT_DIR / 'metrics_summary.json', orient='records', indent=2, force_ascii=False)
per_question_df.to_json(OUT_DIR / 'metrics_by_question.jsonl', orient='records', lines=True, force_ascii=False)

summary_df

## 5. Visual results

In [ ]:
cols = ['hit@10', 'recall@10', 'mrr@10', 'precision@10']
ax = summary_df.set_index('method')[cols].plot(kind='bar', figsize=(12, 5), title='Fast retrieval metrics @10')
ax.set_ylim(0, 1.0)
ax.set_ylabel('score')
plt.xticks(rotation=20, ha='right')
plt.tight_layout()
plt.show()

## 6. Error analysis

In [ ]:
hybrid_method = f'hybrid_lsa_tfidf_alpha_{HYBRID_ALPHA:.2f}'
hybrid_misses = per_question_df[(per_question_df['method'] == hybrid_method) & (per_question_df['hit@10'] == 0)]
print('Hybrid miss@10:', len(hybrid_misses))
hybrid_misses[['query_id', 'num_relevant_chunks', 'question', 'answer', 'recall@10', 'mrr@10']].head(30)